In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_cost_monitoring/cluster_node_type"
tgt_silver_table = "data_governance.silver_cost_monitoring.cluster_node"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df = df.select(
    col("account_id"),
    lower(col("node_type")).alias("node_type"),
    col("core_count").cast("int"),
    col("memory_mb").cast("int"),
    col("gpu_count").cast("int"),
    to_timestamp("load_timestamp").alias("load_timestamp")
)

df = df.filter(
    (col("node_type").isNotNull()) &
    (col("core_count") > 0) &
    (col("memory_mb") > 0) &
    (col("gpu_count") >= 0)
)


df = df.withColumn(
    "instance_size",
    regexp_extract("node_type", "\\.(.*)", 1)
)

df = df.withColumn(
    "memory_gb",
    col("memory_mb") / 1024
)

df = df.withColumn(
    "is_gpu_node",
    when(col("gpu_count") > 0, True).otherwise(False)
)

In [0]:
df.limit(20).display()

In [0]:

# Writing with Liquid Clustering
df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("instance_size") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_cost_monitoring.cluster_node;